# Текстовые эмбеддинги и семантический поиск

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Текстовые эмбеддинги и семантический поиск».

## Что делает метод

Текстовый эмбеддинг — это представление фрагмента текста числовым вектором так, что близким по смыслу текстам соответствуют близкие векторы. Это позволяет искать документы по содержанию, а не по точному совпадению слов: запрос «влияние температуры на урожайность» найдёт статью «как нагрев почвы сказывается на сборе зерна», даже если ни одно слово не совпало.

В этом блокноте мы построим эмбеддинги небольшого набора научных аннотаций, выполним семантический поиск по запросу и визуализируем смысловое пространство. Чтобы блокнот запускался за секунды без тяжёлых моделей, мы используем классический эмбеддинг TF-IDF с понижением размерности (латентно-семантический анализ). Принцип семантического поиска при этом тот же, что и у нейросетевых эмбеддингов.

Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте карту научных публикаций, где каждая статья — точка, а близкие по теме работы расположены рядом. Статьи о «влиянии температуры на урожайность» и «нагреве почвы и сборе зерна» — соседние точки, хотя ни одно слово в их названиях не совпадает. А работы по астрофизике и агрономии — далеко на разных краях карты.

Именно такую карту строит **эмбеддинг**: он переводит текст в вектор (набор чисел) так, чтобы **смысловая близость становилась геометрической**. Когда карта построена, найти релевантные работы по запросу — значит найти ближайших соседей точки «запрос» на этой карте.

Весь блокнот посвящён тому, чтобы вы увидели эту карту своими глазами.

**Ключевые термины, которые встретятся в блокноте:**

- **TF-IDF** — классический способ описать документ: каждое слово получает вес, пропорциональный частоте в документе (TF) и обратно пропорциональный тому, в скольких документах оно встречается (IDF — редкое слово информативнее частого).
- **LSA (латентно-семантический анализ)** — метод сжатия разреженных частотных векторов в плотный эмбеддинг: через разложение матрицы по сингулярным числам (SVD) он выделяет скрытые «темы», на которые откликаются похожие документы.
- **Нормировка векторов** — приведение каждого вектора к единичной длине, чтобы длинные и короткие тексты сравнивались наравне. После нормировки скалярное произведение двух векторов равно **косинусной близости**.
- **Косинусная близость** — число от -1 до 1, показывающее, насколько совпадают направления двух векторов; для нормированных эмбеддингов 1 = идентичный смысл, 0 = нет связи.

## 1. Установка библиотек

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Возьмём небольшую коллекцию коротких научных аннотаций из нескольких разных областей. Корпус синтетический, но воспроизводит типичную ситуацию: тексты группируются по темам, а внутри темы используются разные слова.

In [ ]:
import numpy as np
import pandas as pd

corpus = [
    "Измерение температуры почвы и его влияние на урожайность пшеницы",
    "Нагрев грунта в засушливый период снижает сбор зерновых культур",
    "Внесение азотных удобрений повышает урожай ячменя на опытных полях",
    "Численное моделирование турбулентного течения вязкой жидкости в трубе",
    "Расчёт поля скоростей потока газа методом конечных элементов",
    "Уравнения Навье-Стокса и устойчивость ламинарного течения",
    "Секвенирование генома бактерии и анализ кодирующих участков",
    "Экспрессия генов клетки в ответ на тепловой стресс организма",
    "Мутации ДНК и их влияние на синтез белка в живой клетке",
    "Спектральный анализ излучения далёких галактик в телескоп",
    "Измерение красного смещения звёзд для оценки расширения Вселенной",
    "Наблюдение сверхновых и определение расстояний в космологии",
]
labels = (["агрономия"] * 3 + ["гидродинамика"] * 3
          + ["генетика"] * 3 + ["астрофизика"] * 3)

data = pd.DataFrame({"текст": corpus, "область": labels})
print(f"Документов в корпусе: {len(data)}")
data

## 4. Применение метода

### Шаг 1. Построение эмбеддингов (TF-IDF + LSA)

Конвейер состоит из трёх шагов, выполняемых последовательно:

1. **TF-IDF** — переводит каждый документ в разреженный вектор размерности «словарь»: большинство значений нулевые (слово отсутствует). Редкие информативные слова получают больший вес.
2. **TruncatedSVD** (латентно-семантический анализ) — сжимает разреженный вектор в плотный эмбеддинг из 5 чисел: выделяет скрытые «темы», объединяющие похожие документы, даже если их словари не пересекаются.
3. **Normalizer** — приводит каждый вектор к единичной длине, чтобы косинусная близость была эквивалентна скалярному произведению (это ускоряет поиск).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

# TF-IDF переводит тексты в частотные векторы,
# TruncatedSVD сжимает их в плотный эмбеддинг,
# Normalizer приводит векторы к единичной длине (для косинусной близости).
embedder = make_pipeline(
    TfidfVectorizer(),
    TruncatedSVD(n_components=5, random_state=42),
    Normalizer(),
)
embeddings = embedder.fit_transform(corpus)
print(f"Размер эмбеддинга одного документа: {embeddings.shape[1]} чисел")
print(f"Матрица эмбеддингов: {embeddings.shape}")

### Шаг 2. Семантический поиск

Запрос превращается в эмбеддинг той же моделью и сравнивается со всеми документами. Документы ранжируются по косинусной близости.

**Ключевой момент:** запрос «как жара влияет на сбор урожая» не содержит ни одного слова из документов коллекции (там «температура», «нагрев», «урожайность», «зерновые»). TF-IDF с поиском по точным словам этот запрос не найдёт. LSA-эмбеддинг улавливает общую тему — «тепловое воздействие на растениеводство» — и находит нужные документы.

In [ ]:
def semantic_search(query, top_k=3):
    """Возвращает наиболее близкие по смыслу документы."""
    q_vec = embedder.transform([query])
    # Эмбеддинги нормированы, поэтому скалярное произведение
    # равно косинусной близости.
    scores = embeddings @ q_vec[0]
    order = np.argsort(scores)[::-1][:top_k]
    return pd.DataFrame({
        "близость": scores[order].round(3),
        "область": [labels[i] for i in order],
        "текст": [corpus[i] for i in order],
    })


query = "как жара влияет на сбор урожая"
print(f"Запрос: «{query}»\n")
semantic_search(query)

### Шаг 3. Визуализация смыслового пространства

Эмбеддинги многомерны; чтобы увидеть их структуру, спроецируем их на плоскость. Документы из одной области должны образовать обособленные группы. Ячейка ниже также добавляет на карту **точку запроса** — посмотрите, куда она попадёт относительно тематических групп.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD as TSVD2

# Запрос для отображения на карте.
query = "как жара влияет на сбор урожая"
q_vec = embedder.transform([query])
q_coords_src = TruncatedSVD(n_components=2, random_state=0)

# Проекция корпуса + запроса совместно для сопоставимости координат.
all_vecs = np.vstack([embeddings, q_vec])
pca2 = TSVD2(n_components=2, random_state=0)
all_coords = pca2.fit_transform(all_vecs)
coords = all_coords[:-1]       # координаты документов
q_coord = all_coords[-1]       # координата запроса

fig, ax = plt.subplots(figsize=(11, 7))
palette = {area: VIZ["series"][i]
           for i, area in enumerate(sorted(set(labels)))}
for area in palette:
    mask = np.array(labels) == area
    ax.scatter(coords[mask, 0], coords[mask, 1], s=150,
               color=palette[area], label=area,
               edgecolors="white", linewidths=0.8, zorder=3)
for i, (x, y) in enumerate(coords):
    ax.annotate(str(i), (x, y), fontsize=9, ha="center", va="center",
                color="white", weight="bold", zorder=4)

# Точка запроса.
ax.scatter(*q_coord, s=200, color=VIZ["series"][2], marker="*",
           edgecolors=VIZ["edge"], linewidths=1.0, label="Запрос", zorder=5)
ax.annotate(f"Запрос:\n«{query[:30]}…»", q_coord,
            textcoords="offset points", xytext=(10, 8),
            fontsize=9, color=VIZ["node_text"])

ax.set_title("Карта смыслового пространства документов\n"
             "(звезда — запрос; кружки — документы, пронумерованы по таблице)")
ax.set_xlabel("Смысловая ось 1")
ax.set_ylabel("Смысловая ось 2")
ax.legend(title="Область", loc="best", fontsize=9)
fig.tight_layout()
plt.show()

### Как читать карту смыслового пространства

- Каждый **кружок** — один документ из коллекции. Число внутри — его порядковый номер из таблицы выше.
- **Цвет** кружка соответствует научной области (из списка `labels`). Если документы одного цвета образуют компактную группу — эмбеддинг правильно улавливает тематику.
- **Звезда** — запрос. Посмотрите, к кружкам какой области она расположена ближе всего: это именно те документы, которые функция `semantic_search` должна была вернуть первыми.
- **Оси** не имеют прямой содержательной интерпретации: это первые два главных компонента эмбеддингов. Смысл несут только **расстояния** между точками.
- Если все точки расположены хаотично без группировки — либо коллекция однородна, либо размерность эмбеддинга (5) слишком мала для улавливания нюансов. Попробуйте `n_components=10` в конвейере.

### «Ага»-эксперимент: матрица попарной близости

Вычислим и отобразим матрицу близости всех документов друг к другу. Яркая клетка (i, j) означает, что документы i и j близки по смыслу. На диагонали всегда максимум (документ близок к себе). Блочная структура вне диагонали — признак тематических групп.

In [ ]:
sim_matrix = embeddings @ embeddings.T
n_docs = len(corpus)
short_labels = [f"{i}: {t[:22]}…" for i, t in enumerate(corpus)]

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(sim_matrix, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(n_docs))
ax.set_yticks(range(n_docs))
ax.set_xticklabels([str(i) for i in range(n_docs)], fontsize=9)
ax.set_yticklabels(short_labels, fontsize=8)
ax.set_title("Матрица попарной косинусной близости документов\n"
             "(чем ярче клетка, тем ближе документы по смыслу)", fontsize=13)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.12,
             label="Косинусная близость")
ax.grid(False)
fig.tight_layout()
plt.show()

### Как читать матрицу близости

- **Яркие блоки вдоль диагонали** — документы внутри одной тематической группы похожи друг на друга.
- **Тёмные клетки вне блоков** — разные тематики (агрономия и астрофизика) слабо связаны.
- Найдите самую яркую клетку вне главной диагонали: это пара документов, максимально похожих по смыслу. Проверьте, совпадает ли это с вашим «ручным» ожиданием.
- Если матрица почти равномерно серая (мало контраста) — эмбеддинги не улавливают тематические различия. Увеличьте `n_components` или перейдите на нейросетевые эмбеддинги.

## 5. Интерпретация результата

- **Близость** в выдаче поиска — косинусная мера от -1 до 1; значения около 1 означают высокое смысловое сходство. Поиск находит релевантные документы, даже когда слова запроса и документа не совпадают буквально.
- **Карта смыслового пространства**: документы одной научной области расположены рядом и образуют группы. Это подтверждает, что эмбеддинг улавливает смысл, а не поверхностные признаки текста.
- Размерность эмбеддинга — компромисс: слишком малая теряет смысловые нюансы, слишком большая возвращает к разреженному частотному представлению.

Помните: качество поиска ограничено корпусом, на котором обучена модель. TF-IDF не понимает синонимов, отсутствующих в корпусе; нейросетевые эмбеддинги (модели типа SBERT) обобщают лучше, но требуют больше ресурсов. Принцип семантического поиска при этом одинаков.

## 6. Попробуйте на своих данных

Замените демонстрационный корпус своими текстами — аннотациями статей, описаниями экспериментов, фрагментами протоколов.

1. Подготовьте список строк (по одному тексту на элемент) или таблицу с колонкой текста.
2. Снимите комментарии в ячейке ниже и укажите путь к файлу либо вставьте тексты напрямую.
3. Выполните ячейки раздела 4: эмбеддинги, поиск и карта перестроятся для ваших данных.

## 7. Поэкспериментируйте сами

**Эксперимент 1. Размерность эмбеддинга.**
В конвейере измените `TruncatedSVD(n_components=5)` на `n_components=2` (очень сжато) или `n_components=10` (больше нюансов). Как меняется качество поиска и структура карты?

**Эксперимент 2. Синонимичные запросы.**
Попробуйте несколько перефразировок одного запроса: «засуха и пшеница», «климат и агрономия», «высокая температура и урожай». Меняется ли топ-3? Насколько стабилен результат?

**Эксперимент 3. Добавьте свои документы.**
Добавьте в `corpus` и `labels` 3–5 своих аннотаций из вашей области. Перестройте эмбеддинги и карту. Куда попадут ваши документы? Находит ли поиск нужное?

**Эксперимент 4. Сравните TF-IDF vs. LSA.**
Уберите `TruncatedSVD` из конвейера (замените на прямое `Normalizer`). Как изменится качество поиска по перефразированным запросам? Это наглядно показывает, что LSA добавляет семантическое обобщение поверх простого подсчёта слов.

**Эксперимент 5. Матрица близости.**
Найдите в матрице два наиболее похожих документа (максимум вне диагонали). Прочитайте их тексты: действительно ли они близки по смыслу? Найдите наименее похожую пару (минимум).

## Краткие выводы

- Эмбеддинг превращает текст в вектор так, что **смысловая близость = геометрическая близость**. Карта смыслового пространства и матрица близости делают это наглядным.
- TF-IDF + LSA — быстрый и надёжный базовый метод; нейросетевые энкодеры (SBERT, специализированные для науки) дают лучшее качество на синонимах и парафразах.
- Косинусная близость 1 — идентичный смысл, 0 — нет связи. Всегда проверяйте результаты на размеченных примерах из вашей области.
- Для поиска по большим коллекциям (тысячи документов) используйте векторные базы данных (FAISS, Chroma): принцип тот же, скорость — несравнимо выше.
- Матрица попарной близости полезна для обнаружения дублирующихся работ и для понимания тематической структуры коллекции.

In [ ]:
# --- Шаблон загрузки своих данных ---
# raw = pd.read_csv("my_texts.csv")        # ваш файл
# corpus = raw["текст"].astype(str).tolist()
# labels = raw["категория"].astype(str).tolist()   # необязательно
#
# embeddings = embedder.fit_transform(corpus)
# print(semantic_search("ваш поисковый запрос"))
# Далее повторите ячейки визуализации раздела 4.